# 01 · Load spikes from NWB — **Julia / Judananas**

Side-by-side with `01_load_spikes_python.ipynb`: same session, same steps, same printed checks.

Session `216301_20200521-probe0`: Neuropixels in mPFC, aversive conditioning task (542 units, 570 trials).

In [ ]:
using Pkg
Pkg.activate(".")        # julia/Project.toml: Judananas (from GitHub), HDF5, Plots
Pkg.instantiate()

In [ ]:
using Judananas
using HDF5
using Plots

NWB = "/Volumes/MorseSSD/NPX_Database/frozen_Aversion_Nat_Submission/216301_20200521-probe0.nwb"

## 1. Load the file

`load_file` works like `nap.load_file`: it lists the objects in the file, and each one is read when you first access it.
HDF5.jl must be loaded too, because it does the reading.

In [ ]:
@time data = load_file(NWB)

In [ ]:
units = data["units"]

## 2. Unit metadata

Every column of the units table is already metadata, including the ones pynapple misses in this file
(FS/RS, waveform features, `quality`), plus the columns of each unit's electrode (`location`, AP/DV/ML).
We only add the same two derived columns as in the Python notebook.

In [ ]:
units[:cell_type] = [fs == 1 ? "FS" : rs == 1 ? "RS" : "other" for (fs, rs) in zip(units[:FS], units[:RS])]
units[:isi_fp] = [q[2] for q in units[:quality]]        # ISI violation false-positive rate
units

## 3. Trials

In [ ]:
trials = data["trials"]

## 4. Checks shared with the Python notebook

These numbers should be identical in both notebooks.

In [ ]:
in_trials = restrict(units, trials)
c = count(units, 0.05; ep = trials)

println("units: ", length(units))
println("total spikes: ", sum(length, units))
println("time support: ", collect(time_support(units)))
println("trials: ", length(trials), ", total ", round(tot_length(trials), digits = 3), " s")
println("spikes in trials: ", sum(length, in_trials))
println("count during trials: ", size(c), ", sum ", sum(values(c)))

## 5. Select units from metadata

In [ ]:
pl5 = filter(u -> u.location == "PL5" && u.rate > 1, units)
println("PL5 units > 1 Hz: ", length(pl5))
sort([loc => count(==(loc), units[:location]) for loc in unique(units[:location])], by = last, rev = true)

## 6. Raster of PL5 units, 600–630 s, trials shaded

In [ ]:
win = IntervalSet(600, 630)
plot(restrict(pl5, win), size = (900, 450), title = "PL5 units > 1 Hz, 600–630 s",
     left_margin = 5Plots.mm, bottom_margin = 5Plots.mm)
plot!(intersect(trials, win), color = :orange)

## 7. One unit aligned to sound trials

The PL5 unit with the most spikes inside trials, aligned to trial start, from -1 to 3 s.

In [ ]:
sound_trials = trials[findall(trials[:sound] .== 1)]
in_pl5 = restrict(pl5, trials)
k = argmax(length.(in_pl5))
uid = ids(pl5)[k]
println("unit ", uid, " | sound trials: ", length(sound_trials))

pe = compute_perievent(pl5[k], starts(sound_trials), (-1, 3))
counts = count(pe, 0.05)
psth = Tsd(times(counts), vec(sum(values(counts), dims = 2)) ./ (length(pe) * 0.05))

p1 = plot(pe, yguide = "Trial", title = "Unit $uid", xguide = "")
plot!(p1, IntervalSet(0, 2), color = :orange)
p2 = plot(psth, linewidth = 2, xguide = "Time from trial start (s)", yguide = "Rate (Hz)")
plot!(p2, IntervalSet(0, 2), color = :orange)
display(plot(p1, p2, layout = (2, 1), size = (700, 600), link = :x))
println("PSTH mean rate: ", round(sum(values(psth)) / length(psth), digits = 4), " Hz")

## 8. Export a per-unit summary and compare it with pynapple

In [ ]:
open("units_summary_julia.csv", "w") do io
    println(io, "id,n_spikes,rate,n_spikes_trials,location,cell_type")
    for i in 1:length(units)
        println(io, join((ids(units)[i], length(units[i]), rates(units)[i],
                          length(in_trials[i]), units[:location][i], units[:cell_type][i]), ","))
    end
end

# Compare with the CSV written by the Python notebook (run it first)
function read_summary(path)
    rows = split.(readlines(path)[2:end], ",")
    (id = parse.(Int, getindex.(rows, 1)), n = parse.(Int, getindex.(rows, 2)),
     rate = parse.(Float64, getindex.(rows, 3)), n_trials = parse.(Int, getindex.(rows, 4)),
     location = String.(getindex.(rows, 5)), cell_type = String.(getindex.(rows, 6)))
end

PY_CSV = joinpath("..", "python", "units_summary_python.csv")
if isfile(PY_CSV)
    jl, py = read_summary("units_summary_julia.csv"), read_summary(PY_CSV)
    println("ids identical:              ", jl.id == py.id)
    println("spike counts identical:     ", jl.n == py.n)
    println("spikes in trials identical: ", jl.n_trials == py.n_trials)
    println("locations identical:        ", jl.location == py.location)
    println("cell types identical:       ", jl.cell_type == py.cell_type)
    println("max |rate difference|:      ", maximum(abs.(jl.rate .- py.rate)))
else
    println("No ../python/units_summary_python.csv found: nothing to compare.")
end